In [2]:
import os
import pandas as pd
print(os.getcwd())

<LOCAL_PROJECT_PATH>\notebooks


In [3]:
columns = [
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]

data_path = "../data/raw/UserBehavior.csv"

df = pd.read_csv(
    data_path,
    names=columns,
    nrows=100_000
)

df.head()

,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [4]:
print("Shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nBehavior counts:")
print(df["behavior_type"].value_counts())

print("\nUnique users:")
print(df["user_id"].nunique())

print("\nUnique items:")
print(df["item_id"].nunique())

print("\nUnique categories:")
print(df["category_id"].nunique())

Shape:
(100000, 5)

Data types:
user_id          int64
item_id          int64
category_id      int64
behavior_type      str
timestamp        int64
dtype: object

Missing values:
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64

Behavior counts:
behavior_type
pv      89709
cart     5446
fav      2744
buy      2101
Name: count, dtype: int64

Unique users:
983

Unique items:
64467

Unique categories:
3128


In [5]:
df["datetime"] = (
    pd.to_datetime(
        df["timestamp"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Shanghai")
)

print(df[["timestamp", "datetime"]].head())

print("\nEarliest time:")
print(df["datetime"].min())

print("\nLatest time:")
print(df["datetime"].max())

    timestamp                  datetime
0  1511544070 2017-11-25 01:21:10+08:00
1  1511561733 2017-11-25 06:15:33+08:00
2  1511572885 2017-11-25 09:21:25+08:00
3  1511593493 2017-11-25 15:04:53+08:00
4  1511596146 2017-11-25 15:49:06+08:00

Earliest time:
2017-09-11 16:16:39+08:00

Latest time:
2017-12-03 23:59:23+08:00


In [6]:
start_time = pd.Timestamp(
    "2017-11-25 00:00:00",
    tz="Asia/Shanghai"
)

end_time = pd.Timestamp(
    "2017-12-04 00:00:00",
    tz="Asia/Shanghai"
)

valid_time_mask = (
    (df["datetime"] >= start_time)
    &
    (df["datetime"] < end_time)
)

print("Valid rows:")
print(valid_time_mask.sum())

print("\nInvalid rows:")
print((~valid_time_mask).sum())

print("\nInvalid ratio:")
print((~valid_time_mask).mean())

print("\nExample invalid rows:")
display(
    df.loc[
        ~valid_time_mask,
        ["user_id", "item_id", "behavior_type", "datetime"]
    ]
    .sort_values("datetime")
    .head(20)
)

Valid rows:
99956

Invalid rows:
44

Invalid ratio:
0.00044

Example invalid rows:


,user_id,item_id,behavior_type,datetime
4170,1000169,1328010,pv,2017-09-11 16:16:39+08:00
97122,1004259,3734552,pv,2017-11-17 21:22:30+08:00
18408,1000801,1034143,pv,2017-11-20 22:15:14+08:00
18574,1000807,1662243,pv,2017-11-23 02:03:21+08:00
20069,1000872,2135785,pv,2017-11-23 18:19:37+08:00
20070,1000872,764155,pv,2017-11-23 18:19:44+08:00
32760,1001448,4711795,pv,2017-11-23 20:55:05+08:00
93517,1004075,1912136,pv,2017-11-23 22:47:54+08:00
83059,1003613,3089876,pv,2017-11-24 06:18:14+08:00
92816,1004035,2080690,pv,2017-11-24 12:59:40+08:00


In [7]:
df = df.loc[valid_time_mask].copy()

print("Shape after filtering:")
print(df.shape)

print("\nTime range after filtering:")
print(df["datetime"].min())
print(df["datetime"].max())

Shape after filtering:
(99956, 6)

Time range after filtering:
2017-11-25 00:00:17+08:00
2017-12-03 23:59:23+08:00


In [8]:
print("Behavior types:")
print(df["behavior_type"].value_counts())

print("\nUnique behavior types:")
print(df["behavior_type"].unique())

Behavior types:
behavior_type
pv      89665
cart     5446
fav      2744
buy      2101
Name: count, dtype: int64

Unique behavior types:
<StringArray>
['pv', 'fav', 'buy', 'cart']
Length: 4, dtype: str


In [9]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:")
print(duplicate_count)

print("\nDuplicate ratio:")
print(duplicate_count / len(df))

Duplicate rows:
0

Duplicate ratio:
0.0


In [10]:
user_activity = (
    df.groupby("user_id")
      .size()
      .rename("behavior_count")
)

print("User activity statistics:")
print(
    user_activity.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("\nTop 10 most active users:")
print(
    user_activity
    .sort_values(ascending=False)
    .head(10)
)

User activity statistics:
count    983.000000
mean     101.684639
std       90.640447
min        3.000000
50%       74.000000
75%      135.500000
90%      219.000000
95%      280.800000
99%      447.260000
max      539.000000
Name: behavior_count, dtype: float64

Top 10 most active users:
user_id
1000664    539
1003766    528
1000718    504
1003707    504
1002909    501
1000040    466
1000436    463
1004196    462
1000417    456
1004050    453
Name: behavior_count, dtype: int64


In [11]:
user_unique_items = (
    df.groupby("user_id")["item_id"]
      .nunique()
      .rename("unique_item_count")
)

print("Unique item statistics per user:")

print(
    user_unique_items.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("\nTop 10 users by unique items:")

print(
    user_unique_items
    .sort_values(ascending=False)
    .head(10)
)

Unique item statistics per user:
count    983.000000
mean      76.998983
std       66.876707
min        3.000000
50%       56.000000
75%      104.000000
90%      161.000000
95%      211.800000
99%      327.180000
max      419.000000
Name: unique_item_count, dtype: float64

Top 10 users by unique items:
user_id
1000664    419
1003707    394
1000417    393
1001232    372
1003766    358
1004050    357
1002763    342
1000718    331
1003152    330
1000228    328
Name: unique_item_count, dtype: int64
